# Retrieval Query Strategy Comparison (Qdrant)

This notebook evaluates three retrieval query strategies over 10 random samples from `evaluation.json` using Qdrant at `localhost:6333`.

Strategies:
1. Full file content as query.
2. Regex/string-based intelligent query generation.
3. Static-analysis-driven targeted queries using flake8 + pylint + AST doc-format checks.

In [2]:
import ast
import json
import random
import re
import subprocess
from pathlib import Path

import pandas as pd
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client import models

RANDOM_SEED = 42
SAMPLE_SIZE = 10
TOP_K = 10
COLLECTION_NAME = "guideline_embeddings"
MODEL_NAME = "BAAI/bge-large-en-v1.5"

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
EVAL_PATH = DATA_DIR / "evaluation.json"
EVAL_FILES_DIR = DATA_DIR / "evaluation_files"

print(f"Project root: {PROJECT_ROOT}")
print(f"Evaluation file: {EVAL_PATH}")

c:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project
Evaluation file: C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project\data\processed\evaluation.json


In [3]:
# Load evaluation data and select 10 random samples
with EVAL_PATH.open("r", encoding="utf-8") as f:
    evaluation_data = json.load(f)

random.seed(RANDOM_SEED)
sampled_entries = random.sample(evaluation_data, k=min(SAMPLE_SIZE, len(evaluation_data)))

def resolve_source_file(entry):
    source_path = Path(entry["source_file"])
    if source_path.is_absolute():
        return source_path
    return EVAL_FILES_DIR / source_path.name if not (EVAL_FILES_DIR / source_path).exists() else EVAL_FILES_DIR / source_path

for entry in sampled_entries:
    file_path = resolve_source_file(entry)
    entry["resolved_source_file"] = str(file_path)
    entry["file_text"] = file_path.read_text(encoding="utf-8", errors="ignore")

print(f"Loaded {len(evaluation_data)} total entries")
print(f"Sampled {len(sampled_entries)} entries")
pd.DataFrame([
    {
        "id": e["id"],
        "repo": e["repo"],
        "source_path": e["source_path"],
        "resolved_source_file": e["resolved_source_file"]
    }
    for e in sampled_entries
])

Loaded 97 total entries
Sampled 10 entries


,id,repo,source_path,resolved_source_file
0,synthetic-sklearn_PR_24,kannan-dedsec/synthetic-sklearn,custom_transformer.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...
1,synthetic-django_PR_36,kannan-dedsec/synthetic-django,urls.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...
2,synthetic-django_PR_24,kannan-dedsec/synthetic-django,managers.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...
3,synthetic-sklearn_PR_38,kannan-dedsec/synthetic-sklearn,tests/test_models.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...
4,synthetic-fastapi_PR_38,kannan-dedsec/synthetic-fastapi,tests/test_users.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...
5,synthetic-fastapi_PR_34,kannan-dedsec/synthetic-fastapi,schemas/item.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...
6,synthetic-fastapi_PR_31,kannan-dedsec/synthetic-fastapi,routers/auth.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...
7,synthetic-django_PR_39,kannan-dedsec/synthetic-django,views.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...
8,synthetic-django_PR_35,kannan-dedsec/synthetic-django,tests/test_views.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...
9,synthetic-sklearn_PR_30,kannan-dedsec/synthetic-sklearn,hyperparameter_tuning.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...


In [4]:
# Connect to Qdrant and load embedding model
client = QdrantClient(url="http://localhost:6333")
collection_info = client.get_collection(COLLECTION_NAME)
print(f"Connected to Qdrant. Collection: {COLLECTION_NAME}")
print(f"Points count: {collection_info.points_count}")

model = SentenceTransformer(MODEL_NAME)
print(f"Loaded model: {MODEL_NAME}")

Connected to Qdrant. Collection: guideline_embeddings
Points count: 505


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1086.09it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: BAAI/bge-large-en-v1.5


In [5]:
COMMON_SOURCE_TYPES = ["pep8", "flake8", "pylint", "ruff"]

def repo_family(repo_name):
    repo_lower = repo_name.lower()
    for family in ["django", "fastapi", "flask", "pandas", "sklearn", "scikit-learn"]:
        if family in repo_lower:
            return "scikit-learn" if family == "sklearn" else family
    return None

def build_query_filter(repo_name):
    family = repo_family(repo_name)
    should_conditions = [
        models.FieldCondition(key="source_type", match=models.MatchValue(value=s))
        for s in COMMON_SOURCE_TYPES
    ]
    if family:
        should_conditions.insert(
            0,
            models.FieldCondition(
                key="source_type",
                match=models.MatchValue(value=f"{family}_guidelines"),
            ),
        )
        should_conditions.insert(
            1,
            models.FieldCondition(
                key="source_type",
                match=models.MatchValue(value=f"{family}_review_comment"),
            ),
        )
    return models.Filter(should=should_conditions)

def retrieve_guidelines(query_text, repo_name, top_k=TOP_K):
    query_vector = model.encode(query_text).tolist()
    response = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=build_query_filter(repo_name),
        limit=top_k,
    )
    return response.points

print("Retrieval helpers ready")

Retrieval helpers ready


In [6]:
# Strategy 1: Full file as query
def query_strategy_full_file(entry):
    text = entry["file_text"]
    header = f"Repository: {entry['repo']} | File: {entry['source_path']}\n"
    return header + text

# Strategy 2: Regex/string-based intelligent query
def query_strategy_regex_intelligent(entry):
    text = entry["file_text"]
    lines = text.splitlines()

    imports = [ln.strip() for ln in lines if re.match(r"^\s*(import|from)\s+", ln)]
    camel_case_identifiers = set(re.findall(r"\b[a-z]+[A-Z][A-Za-z0-9_]*\b", text))
    mutable_defaults = re.findall(r"def\s+\w+\([^)]*=\s*(\[\]|\{\}|dict\(|list\()", text)

    indent_issues = 0
    for ln in lines:
        stripped = ln.lstrip(" ")
        if not stripped or ln.startswith("#"):
            continue
        leading_spaces = len(ln) - len(stripped)
        if leading_spaces > 0 and leading_spaces % 4 != 0:
            indent_issues += 1

    docstring_markers = len(re.findall(r"'''|\"\"\"", text))

    signals = [
        f"repo family: {repo_family(entry['repo'])}",
        f"import statements count: {len(imports)}",
        f"camelCase identifiers: {', '.join(sorted(list(camel_case_identifiers))[:8]) or 'none'}",
        f"mutable-default patterns: {len(mutable_defaults)}",
        f"non-4-space indentation lines: {indent_issues}",
        f"docstring markers count: {docstring_markers}",
        "focus on naming_convention, unused_import, indentation, mutable_default, documentation_formatting"
    ]

    return " ; ".join(signals)

print("Strategy 1 and Strategy 2 query builders ready")

Strategy 1 and Strategy 2 query builders ready


In [7]:
# Strategy 3: Static-analysis-driven targeted query
def run_tool(command):
    try:
        result = subprocess.run(command, capture_output=True, text=True, check=False)
        stdout = result.stdout.strip()
        stderr = result.stderr.strip()
        return result.returncode, stdout, stderr
    except FileNotFoundError:
        return 127, "", f"Command not found: {' '.join(command)}"

def parse_flake8(output):
    findings = []
    for line in output.splitlines():
        m = re.match(r"^(.*?):(\d+):(\d+):\s*([A-Z]\d+)\s*(.*)$", line)
        if not m:
            continue
        findings.append({
            "tool": "flake8",
            "line": int(m.group(2)),
            "col": int(m.group(3)),
            "code": m.group(4),
            "message": m.group(5),
        })
    return findings

def parse_pylint(output):
    findings = []
    for line in output.splitlines():
        m = re.match(r"^(.*?):(\d+):(\d+):\s*([A-Z]\d+):\s*(.*?)\s*\((.*?)\)$", line)
        if not m:
            continue
        findings.append({
            "tool": "pylint",
            "line": int(m.group(2)),
            "col": int(m.group(3)),
            "code": m.group(4),
            "message": m.group(5),
            "symbol": m.group(6),
        })
    return findings

def analyze_doc_formatting(source_text):
    """Violation #5 (documentation_formatting): detect missing/short docstrings and complexity context."""
    signals = []
    try:
        tree = ast.parse(source_text)
    except SyntaxError as exc:
        return [{"type": "syntax_error", "message": str(exc)}]

    complex_nodes = (
        ast.If, ast.For, ast.While, ast.Try, ast.With, ast.Match,
        ast.ListComp, ast.DictComp, ast.SetComp, ast.GeneratorExp
    )

    for node in ast.walk(tree):
        if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            continue

        node_type = "async_function" if isinstance(node, ast.AsyncFunctionDef) else ("function" if isinstance(node, ast.FunctionDef) else "class")
        doc = ast.get_docstring(node)
        missing = doc is None
        too_small = False
        if doc is not None:
            word_count = len(doc.split())
            too_small = word_count < 5

        has_decorators = bool(getattr(node, "decorator_list", []))
        complexity_count = sum(1 for child in ast.walk(node) if isinstance(child, complex_nodes))
        has_complex_constructs = complexity_count >= 3

        if missing or too_small:
            signals.append({
                "type": "documentation_formatting",
                "node_type": node_type,
                "name": node.name,
                "line": getattr(node, "lineno", None),
                "missing_docstring": missing,
                "too_small_docstring": too_small,
                "has_decorators": has_decorators,
                "has_async": isinstance(node, ast.AsyncFunctionDef),
                "has_complex_constructs": has_complex_constructs,
                "complexity_count": complexity_count,
            })

    return signals

def static_findings_to_targets(flake8_findings, pylint_findings, doc_signals):
    targets = []

    for item in flake8_findings:
        code = item["code"]
        if code.startswith("F401"):
            targets.append("unused import cleanup")
        if code.startswith("E1"):
            targets.append("indentation and spacing rules")
        if code.startswith("E3") or code.startswith("W2"):
            targets.append("blank lines and whitespace formatting")
        if code.startswith("D"):
            targets.append("documentation and docstring formatting")

    for item in pylint_findings:
        symbol = item.get("symbol", "")
        msg = item.get("message", "").lower()
        if "unused-import" in symbol or "unused import" in msg:
            targets.append("unused import cleanup")
        if "invalid-name" in symbol or "name" in msg:
            targets.append("snake_case naming convention")
        if "dangerous-default-value" in symbol or "default" in msg:
            targets.append("mutable default argument avoidance")
        if "missing-function-docstring" in symbol or "missing-module-docstring" in symbol:
            targets.append("documentation and docstring coverage")

    if doc_signals:
        targets.append("documentation_formatting for missing/too-small docstrings")
        if any(s.get("has_async") for s in doc_signals if isinstance(s, dict)):
            targets.append("docstrings for async function behavior and contracts")
        if any(s.get("has_decorators") for s in doc_signals if isinstance(s, dict)):
            targets.append("docstrings for decorated functions/classes")
        if any(s.get("has_complex_constructs") for s in doc_signals if isinstance(s, dict)):
            targets.append("docstrings for complex control-flow and side effects")

    deduped = sorted(set(targets))
    return deduped

def query_strategy_static(entry):
    file_path = entry["resolved_source_file"]

    flake8_cmd = ["flake8", file_path]
    pylint_cmd = ["pylint", "--output-format=text", file_path]

    flake8_rc, flake8_out, flake8_err = run_tool(flake8_cmd)
    pylint_rc, pylint_out, pylint_err = run_tool(pylint_cmd)

    flake8_findings = parse_flake8(flake8_out)
    pylint_findings = parse_pylint(pylint_out)
    doc_signals = analyze_doc_formatting(entry["file_text"])

    targets = static_findings_to_targets(flake8_findings, pylint_findings, doc_signals)

    if not targets:
        targets = ["general python quality and style guidelines"]

    query_text = " ; ".join([
        f"repo family: {repo_family(entry['repo'])}",
        f"flake8 findings: {len(flake8_findings)} (rc={flake8_rc})",
        f"pylint findings: {len(pylint_findings)} (rc={pylint_rc})",
        f"doc-format signals: {len(doc_signals)}",
        "targeted topics: " + ", ".join(targets),
    ])

    diagnostics = {
        "flake8_rc": flake8_rc,
        "flake8_error": flake8_err,
        "pylint_rc": pylint_rc,
        "pylint_error": pylint_err,
        "flake8_findings": flake8_findings[:15],
        "pylint_findings": pylint_findings[:15],
        "doc_signals": doc_signals[:15],
        "targets": targets,
    }

    return query_text, diagnostics

print("Strategy 3 static-analysis query builder ready")

Strategy 3 static-analysis query builder ready


In [8]:
def run_strategy(name, query_builder, entries):
    rows = []
    diagnostics_rows = []

    for entry in entries:
        if name == "strategy_3_static":
            query_text, diagnostics = query_builder(entry)
            diagnostics_rows.append({
                "sample_id": entry["id"],
                "repo": entry["repo"],
                "diagnostics": diagnostics,
            })
        else:
            query_text = query_builder(entry)

        hits = retrieve_guidelines(query_text, repo_name=entry["repo"], top_k=TOP_K)

        for rank, hit in enumerate(hits, start=1):
            payload = hit.payload or {}
            rows.append({
                "sample_id": entry["id"],
                "repo": entry["repo"],
                "source_path": entry["source_path"],
                "strategy": name,
                "rank": rank,
                "score": float(hit.score),
                "source_type": payload.get("source_type"),
                "category": payload.get("category"),
                "guideline_text": payload.get("text", "")[:220],
                "query_preview": query_text[:260],
            })

    return pd.DataFrame(rows), pd.DataFrame(diagnostics_rows)

results_full, _ = run_strategy("strategy_1_full_file", query_strategy_full_file, sampled_entries)
results_regex, _ = run_strategy("strategy_2_regex_intelligent", query_strategy_regex_intelligent, sampled_entries)
results_static, static_diagnostics = run_strategy("strategy_3_static", query_strategy_static, sampled_entries)

all_results = pd.concat([results_full, results_regex, results_static], ignore_index=True)

print(f"Total retrieved rows: {len(all_results)}")
print("Expected rows = sampled_entries x 3 strategies x top_k =", len(sampled_entries) * 3 * TOP_K)
all_results.head(20)

Total retrieved rows: 300
Expected rows = sampled_entries x 3 strategies x top_k = 300


,sample_id,repo,source_path,strategy,rank,score,source_type,category,guideline_text,query_preview
0,synthetic-sklearn_PR_24,kannan-dedsec/synthetic-sklearn,custom_transformer.py,strategy_1_full_file,1,0.620321,ruff,documentation_formatting,D105: Missing docstring in magic method. Dunde...,Repository: kannan-dedsec/synthetic-sklearn | ...
1,synthetic-sklearn_PR_24,kannan-dedsec/synthetic-sklearn,custom_transformer.py,strategy_1_full_file,2,0.603707,pylint,naming_convention,W3201 (bad-dunder-name): Dunder method name is...,Repository: kannan-dedsec/synthetic-sklearn | ...
2,synthetic-sklearn_PR_24,kannan-dedsec/synthetic-sklearn,custom_transformer.py,strategy_1_full_file,3,0.600728,ruff,naming_convention,N807: Function name should not start and end w...,Repository: kannan-dedsec/synthetic-sklearn | ...
3,synthetic-sklearn_PR_24,kannan-dedsec/synthetic-sklearn,custom_transformer.py,strategy_1_full_file,4,0.595769,ruff,unused_import,Be cautious when auto-removing unused imports ...,Repository: kannan-dedsec/synthetic-sklearn | ...
4,synthetic-sklearn_PR_24,kannan-dedsec/synthetic-sklearn,custom_transformer.py,strategy_1_full_file,5,0.593793,ruff,mutable_default,B006 example fix: Instead of 'def func(items=[...,Repository: kannan-dedsec/synthetic-sklearn | ...
5,synthetic-sklearn_PR_24,kannan-dedsec/synthetic-sklearn,custom_transformer.py,strategy_1_full_file,6,0.591066,ruff,documentation_formatting,D402: First line of the docstring should not b...,Repository: kannan-dedsec/synthetic-sklearn | ...
6,synthetic-sklearn_PR_24,kannan-dedsec/synthetic-sklearn,custom_transformer.py,strategy_1_full_file,7,0.589122,ruff,mutable_default,B006: Do not use mutable data structures (list...,Repository: kannan-dedsec/synthetic-sklearn | ...
7,synthetic-sklearn_PR_24,kannan-dedsec/synthetic-sklearn,custom_transformer.py,strategy_1_full_file,8,0.583870,ruff,unused_import,"For module availability checks, use importlib....",Repository: kannan-dedsec/synthetic-sklearn | ...
8,synthetic-sklearn_PR_24,kannan-dedsec/synthetic-sklearn,custom_transformer.py,strategy_1_full_file,9,0.583245,ruff,documentation_formatting,D417: Missing argument descriptions in the doc...,Repository: kannan-dedsec/synthetic-sklearn | ...
9,synthetic-sklearn_PR_24,kannan-dedsec/synthetic-sklearn,custom_transformer.py,strategy_1_full_file,10,0.582651,ruff,naming_convention,N815: Variable in class scope should not use m...,Repository: kannan-dedsec/synthetic-sklearn | ...


In [9]:
# Comparison summaries
strategy_summary = all_results.groupby("strategy").agg(
    avg_score=("score", "mean"),
    median_score=("score", "median"),
    min_score=("score", "min"),
    max_score=("score", "max"),
    rows=("score", "count"),
).reset_index().sort_values(by="avg_score", ascending=False)

category_distribution = all_results.groupby(["strategy", "category"]).size().reset_index(name="count")
source_type_distribution = all_results.groupby(["strategy", "source_type"]).size().reset_index(name="count")

print("Strategy score summary")
display(strategy_summary)

print("Top categories per strategy")
display(category_distribution.sort_values(["strategy", "count"], ascending=[True, False]).groupby("strategy").head(10))

print("Top source types per strategy")
display(source_type_distribution.sort_values(["strategy", "count"], ascending=[True, False]).groupby("strategy").head(10))

print("Static diagnostics sample")
display(static_diagnostics.head(5))

Strategy score summary


,strategy,avg_score,median_score,min_score,max_score,rows
1,strategy_2_regex_intelligent,0.699604,0.700860,0.658740,0.732468,100
2,strategy_3_static,0.686861,0.689770,0.639498,0.751512,100
0,strategy_1_full_file,0.616698,0.608992,0.573780,0.711967,100


Top categories per strategy


,strategy,category,count
3,strategy_1_full_file,naming_convention,29
4,strategy_1_full_file,unused_import,25
0,strategy_1_full_file,documentation_formatting,21
2,strategy_1_full_file,mutable_default,13
1,strategy_1_full_file,indentation,12
7,strategy_2_regex_intelligent,naming_convention,40
6,strategy_2_regex_intelligent,indentation,24
8,strategy_2_regex_intelligent,unused_import,20
5,strategy_2_regex_intelligent,documentation_formatting,16
9,strategy_3_static,documentation_formatting,51


Top source types per strategy


,strategy,source_type,count
6,strategy_1_full_file,ruff,31
2,strategy_1_full_file,fastapi_review_comment,26
1,strategy_1_full_file,django_review_comment,21
0,strategy_1_full_file,django_guidelines,15
5,strategy_1_full_file,pylint,5
3,strategy_1_full_file,flake8,1
4,strategy_1_full_file,pep8,1
13,strategy_2_regex_intelligent,ruff,42
8,strategy_2_regex_intelligent,django_review_comment,17
9,strategy_2_regex_intelligent,fastapi_review_comment,15


Static diagnostics sample


,sample_id,repo,diagnostics
0,synthetic-sklearn_PR_24,kannan-dedsec/synthetic-sklearn,"{'flake8_rc': 1, 'flake8_error': '', 'pylint_r..."
1,synthetic-django_PR_36,kannan-dedsec/synthetic-django,"{'flake8_rc': 1, 'flake8_error': '', 'pylint_r..."
2,synthetic-django_PR_24,kannan-dedsec/synthetic-django,"{'flake8_rc': 1, 'flake8_error': '', 'pylint_r..."
3,synthetic-sklearn_PR_38,kannan-dedsec/synthetic-sklearn,"{'flake8_rc': 1, 'flake8_error': '', 'pylint_r..."
4,synthetic-fastapi_PR_38,kannan-dedsec/synthetic-fastapi,"{'flake8_rc': 1, 'flake8_error': '', 'pylint_r..."


In [10]:
# Persist outputs for later analysis
output_dir = PROJECT_ROOT / "results" / "retrieval_query_strategy_results_v1"
output_dir.mkdir(parents=True, exist_ok=True)

all_results_path = output_dir / "retrieval_query_strategy_all_results.csv"
summary_path = output_dir / "retrieval_query_strategy_summary.csv"
diagnostics_path = output_dir / "retrieval_query_strategy_static_diagnostics.json"

all_results.to_csv(all_results_path, index=False)
strategy_summary.to_csv(summary_path, index=False)

with diagnostics_path.open("w", encoding="utf-8") as f:
    json.dump(static_diagnostics.to_dict(orient="records"), f, indent=2)

print(f"Saved: {all_results_path}")
print(f"Saved: {summary_path}")
print(f"Saved: {diagnostics_path}")

Saved: C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project\results\retrieval_query_strategy_results_v1\retrieval_query_strategy_all_results.csv
Saved: C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project\results\retrieval_query_strategy_results_v1\retrieval_query_strategy_summary.csv
Saved: C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project\results\retrieval_query_strategy_results_v1\retrieval_query_strategy_static_diagnostics.json
